In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.plot  # submodule is not imported by `import rasterio` alone
import torch
from IPython.display import Image  # For displaying the training GIF

# Use this to load the functions used in this notebook
# You can also 'import deepssf' and call functions as deepssf.function_name
import deepssf
from deepssf import (
    ConvJointModel,
    EarlyStopping,
    ModelParams,
    create_gif,
    filter_steps_by_window,
    fit,
    get_device,
    load_checkpoint,
    load_environmental_layers,
    load_head_weights,
    make_dataloaders,
    make_optimisers,
    negativeLogLikeLoss,
    prepare_movement_df,
    simulate_trajectory,
    validate_next_step_probs,
)

print(f"deepssf {deepssf.__version__}")
print(f"torch {torch.__version__}")

# Setup the script

## Set random seed and the device for training the model

If you have a GPU, this is where it will be specified (automatically).

In [ ]:
# Reproducibility — fix random seeds so results are the same on every run.
# These matter here: make_dataloaders() uses random_split, so without a seed the
# train/val/test partition changes every run and two training configurations
# cannot be compared.
torch.manual_seed(42)
np.random.seed(42)

# DEVICE is capitalised — module-level constant (set once, never reassigned).
# get_device() returns 'cuda' if a GPU is available, 'mps' on Apple Silicon,
# or 'cpu' otherwise.  All tensors and the model are moved to this device.
DEVICE = get_device()
print(f"Device: {DEVICE}")


## Set script paths
If you have cloned/forked/downloaded the GitHub repo (https://github.com/swforrest/deepSSF_package) you shouldn't need to change these, but for your own analyses you will want to set these to point to where your data is and where you want outputs saved.

*All-caps names below are module-level constants — set once here and referenced throughout the notebook without being reassigned.  Python has no built-in constant keyword, so capitalisation is the community convention (PEP 8) to signal "do not change this value".*

In [ ]:
# from rasterio.merge import merge

# src1 = rasterio.open("mapping/feral_pig_aoi_alphaearth_2025-0000000000-0000000000.tif")
# src2 = rasterio.open("mapping/feral_pig_aoi_alphaearth_2025-0000000000-0000003072.tif")

# mosaic, out_transform = merge([src1, src2])

# out_meta = src1.meta.copy()
# out_meta.update({
#     "height": mosaic.shape[1],
#     "width": mosaic.shape[2],
#     "transform": out_transform
# })

# with rasterio.open("mapping/feral_pig_aoi_alphaearth_2025.tif", "w", **out_meta) as dest:
#     dest.write(mosaic)

In [ ]:
DATA_DIR = Path("data")  # location of the bundled sample data
MAPPING_DIR = Path("mapping")  # location of the landcover mapping files
CSV_PATH = DATA_DIR / "pig_all_steps_covs_subset_20260813.csv"  # GPS tracking data, 28 individuals

# NOTE: 's2' and 's2_dir' are RESERVED keys in load_environmental_layers().
# 's2_dir' triggers the monthly-composite loader (load_s2_data), and a plain
# 's2' entry is skipped by the per-file loop — so a single multi-band GeoTIFF
# must be given any *other* name, otherwise nothing is loaded and the function
# raises "raster_transform could not be set".
LAYER_PATHS = {
    # 3-band Sentinel-2 composite: NDVI, NDWI, NDMI (20 m, EPSG:3112)
    # "s2_indices": str(MAPPING_DIR / "feral_pig_aoi_S2_indices_2025-09-12_to_2026-06-23.tif"),
    # Alternative — all 12 S2 bands (12 spatial channels instead of 3, slower to train):
    # "s2_median": str(MAPPING_DIR / "feral_pig_aoi_S2_median_full_2025-09-12_to_2026-06-23.tif"),
    # AlphaEarth embeddings (2025)
    "alphaearth": str(MAPPING_DIR / "feral_pig_aoi_alphaearth_2025_crop.tif"),
}
OUTPUT_DIR   = Path("outputs")                   # figures and model checkpoints go here
SNAPSHOT_DIR = OUTPUT_DIR / "training_snapshots"  # per-batch prediction-map snapshots
OUTPUT_DIR.mkdir(exist_ok=True)

# Import data
The csv file is imported with a timestamp and an x and y coordinate. In our case the data was already in a projected format that is specific to Australia (EPSG:3112), which is the same crs as the raster data which will be imported.

In [ ]:
raw_df = pd.read_csv(CSV_PATH)
print(f"Raw GPS fixes : {len(raw_df):,}")
print(f"Individuals   : {raw_df['name'].nunique()}")

# Sort by the individual ID and timestamp to ensure correct temporal order
raw_df = raw_df.sort_values(by=["name", "t1_"]).reset_index(drop=True)

# Convert the 'time' column to the correct timezone (here Australia/Darwin), then
# drop the offset.  prepare_movement_df() re-parses timestamps with utc=True, so a
# tz-aware Darwin column would be converted straight back to UTC and hour_t1 would
# be 9.5 h out — making the diel covariates wrong.  Localising to Darwin and then
# going tz-naive keeps the local clock time that the sin/cos encodings need.
raw_df["t1_"] = (
        pd.to_datetime(raw_df["t1_"], utc=True)
        .dt.tz_convert("Australia/Darwin")
        .dt.tz_localize(None)
    )

raw_df.head()

## `load_environmental_layers()`

Reads each GeoTIFF listed in LAYER_PATHS, min–max

- Scales every band to [0, 1], and returns:
    * `env_layers`       — dict mapping layer name → NumPy array: 2-D (H × W) for single-band GeoTIFFs, 3-D (bands × H × W) for multi-band ones, where each band becomes its own model input channel
    * `raster_transform` — Affine object that converts pixel (row, col) indices to the projected coordinates (easting, northing) of the raster(s), which should be in the same CRS as the GPS data.  This transform is used whenever we need to translate between pixel space and real-world coordinates.

**Two things to watch for:**

1. The keys `'s2'` and `'s2_dir'` are *reserved* for the monthly Sentinel-2 composite loader — a single GeoTIFF passed under the name `'s2'` is silently skipped (see the note in the paths cell above).
2. The rescaling uses a **single global min/max across all bands** of a file, and computes `(data - lo) / hi` rather than `(data - lo) / (hi - lo)`. Our index raster is `int16` scaled by 10 000 with ±10 000 fill values, so the loaded values span [0, 2] rather than [0, 1] and the three indices are not individually normalised. That is workable (the CNN can rescale internally) but if it hurts training, pre-scale each band yourself and save as `.npy` — non-`.tif` paths in `LAYER_PATHS` are loaded with `np.load(..., mmap_mode='r')` and left untouched, though at least one real `.tif` must remain in the dict so the transform can be set.


In [ ]:
env_layers, raster_transform = load_environmental_layers(LAYER_PATHS)
print("Layers loaded:", list(env_layers.keys()))
for name, arr in env_layers.items():
    print(f"  {name}: shape={arr.shape}, min={arr.min():.3f}, max={arr.max():.3f}")

# Each band of a multi-band layer becomes one CNN input channel, so count bands
# (not layers) here and reuse the total when setting ModelParams below.  Deriving
# it means switching rasters in LAYER_PATHS needs no other edits.
N_SPATIAL_CHANNELS = sum(1 if arr.ndim == 2 else arr.shape[0] for arr in env_layers.values())
print(f"Spatial channels: {N_SPATIAL_CHANNELS}")

###  Plot each spatial covariate with the observed GPS locations overlaid.
This confirms the raster extent covers the animal's home range and gives a visual sense of the landscape the model will learn from.

In [ ]:
# Flatten env_layers into one (2-D array, label) pair per spatial channel so the
# figure adapts to whatever LAYER_PATHS provides (1 layer of 3 bands, 2 layers of
# 1 band, ...).  Band descriptions come from the GeoTIFF metadata where present.
channel_plots = []
for name, arr in env_layers.items():
    if arr.ndim == 2:
        channel_plots.append((arr, name))
        continue
    with rasterio.open(LAYER_PATHS[name]) as src:
        descs = src.descriptions
    for band in range(arr.shape[0]):
        label = descs[band] if descs and descs[band] else f"band {band + 1}"
        channel_plots.append((arr[band], f"{name}: {label}"))

ncol = min(3, len(channel_plots))
nrow = math.ceil(len(channel_plots) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 4.6 * nrow), squeeze=False)

for ax, (arr, label) in zip(axes.flat, channel_plots, strict=False):
    rasterio.plot.show(arr, transform=raster_transform, ax=ax, cmap="viridis")
    # Points rather than a connected line: raw_df holds 28 individuals, so a
    # single line would draw spurious jumps between them.
    ax.plot(
        raw_df["x1_"], raw_df["y1_"],
        "r.", alpha=0.3, markersize=0.7, label="GPS fixes",
    )
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")
    ax.set_title(label)
    ax.legend(loc="upper right", markerscale=10)

# Blank out any unused panels in the grid
for ax in axes.flat[len(channel_plots):]:
    ax.axis("off")

plt.suptitle("Spatial covariates with observed GPS locations", y=1.01)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "spatial_covariates.png"), dpi=100)
plt.show()

## `prepare_movement_df()`
Converts a fixes dataframe (one row per GPS location) into a *step* dataframe (one row per movement between consecutive fixes).

- For each step it computes:
    * `dx`, `dy`         — displacement in projected CRS units (metres)
    * `bearing`          — direction of travel (radians; 0 = north, clockwise positive)
    * `bearing_tm1`      — bearing of the *previous* step, used as a model input so the network can capture directional persistence by estimating a turning angle
    * `dt_hour`          — time elapsed between consecutive fixes (hours)
    * `hour_t1`, `yday_t1` — decimal hour-of-day and day-of-year at the step start location
    * `*_sin1` / `*_cos1`  — sine/cosine encodings of hour and yday (see SCALAR_COLS below)

The trailing underscore on column names (x1_, y1_, t1_, etc.) flags them as "start of step" variables; x2_, y2_, t2_ are the corresponding "end of step" values, following the convention of the `amt` package [@Signer2019-fi]. The last fix for each individual is dropped because no forward step can be formed.

In [ ]:
# The CSV is already in amt step format, but prepare_movement_df() expects one row
# per *fix* and rebuilds the steps itself (it also derives bearing, dt_hour and the
# cyclic time encodings the model needs).  Point it at the step-start columns and
# let it re-form the steps from consecutive fixes.
step_df = prepare_movement_df(
    raw_df, 
    id_col="name", 
    time_col="t1_", 
    x_col="x1_", 
    y_col="y1_"
)
n_dropped = len(raw_df) - len(step_df)
print(f"Movement steps : {len(step_df):,}  ({n_dropped} dropped — last per individual)")
step_df.head()

## `SCALAR_COLS`
These are non-spatial inputs. For processing with convolutional layers, they are converted into spatial windows (rasters/arrays), and stacked with the truly spatial covariates.

The model receives a fixed-size spatial window centred on each step's start location.  To let the network express time-varying habitat preferences (e.g. different nocturnal vs diurnal use), scalar covariates are tiled into additional channels of the same H × W size, so every pixel "knows" the current time context when habitat suitability is estimated.

- `hour_t1_sin1` / `hour_t1_cos1`
    * Sine and cosine of (2π × hour / 24), encoding time-of-day as a *circular* variable.  The circular encoding means 23:00 and 01:00 are treated as adjacent — a plain hour value would have an artificial discontinuity at midnight.

- `yday_t1_sin1` / `yday_t1_cos1`
    * Same circular encoding for day-of-year (1–365), capturing seasonal variation in habitat use (e.g. dry-season vs wet-season movements).

- `dt_hour`
    * Elapsed time between consecutive fixes (hours).  Because GPS collars do not always fix at a perfectly regular rate, including dt_hour lets the movement kernel rescale expected displacement proportionally to the actual time interval rather than assuming a fixed fix rate. Note: dt_hour must be supplied at *simulation* time as well, so that `simulate_trajectory()` produces inputs consistent with training.

In [ ]:
SCALAR_COLS = ["hour_t1_sin1",
               "hour_t1_cos1", 
               "yday_t1_sin1", 
               "yday_t1_cos1", 
               "dt_hour"]


## `WINDOW_SIZE`
The size of the local availability window in cells/pixels; 101 pixels × 20 m = 2020 m across, i.e. 1010 m from the centre location to the edge

In [ ]:
WINDOW_SIZE = 51

## `PIXEL_SIZE`
Raster resolution in CRS units (metres here).  Read straight off `raster_transform`
rather than typed in: the transform is the authority on the grid the model is
trained against, and a hand-entered value that disagrees with it silently
mis-scales the movement kernel — which is parameterised in metres — against a
grid measured in pixels.

In [ ]:
# transform.a is the cell width, transform.e the (negative) cell height
PIXEL_SIZE = abs(raster_transform.a)

if not math.isclose(abs(raster_transform.a), abs(raster_transform.e)):
    raise ValueError(
        f"Non-square pixels ({abs(raster_transform.a)} x {abs(raster_transform.e)}); "
        "the model assumes square cells."
    )
print(f"Pixel size: {PIXEL_SIZE} m  →  window {WINDOW_SIZE} px = "
      f"{WINDOW_SIZE * PIXEL_SIZE:,.0f} m across")

## `BATCH_SIZE`
Number of training samples (movement steps) per training mini-batch

In [ ]:
BATCH_SIZE  = 512

## `OUTPUT_CHANNELS`
Number of convolutional filters (and therefore feature maps) for each convolution layer. More convolutional filters can provide more flexible functions, but will increase the number of parameters and slow down training times.

In [ ]:
OUTPUT_CHANNELS = 8

### Pre-calculate the flattened CNN output size 
This value will be the input to the fully-connected layers in the movement subnetwork, and is calculated here so that ModelParams can be set without hard-coding it.  

- The formula mirrors the layers inside ConvJointModel: 
    * conv (padding=1) keeps the spatial dimension unchanged at each layer
    * maxpool k=2 - the number of max pooling layers (set in the model code and isn't currently editable in this script)
    * s=2 halves the spatial dimension (floor division) at each layer

In [ ]:
dim = WINDOW_SIZE
for _ in range(3):
    dim = math.floor((dim + 2 * 1 - 3) / 1 + 1)  # conv (pad=1 preserves dim)
    dim = math.floor((dim - 2) / 2 + 1)           # maxpool k=2, s=2
DENSE_DIM = OUTPUT_CHANNELS * dim * dim
print(f"After 3× conv+maxpool: dim={dim}  →  dense_dim_in_all={DENSE_DIM}")

## `filter_steps_by_window()` 
Removes steps whose straight-line displacement exceeds half the spatial window (WINDOW_SIZE × PIXEL_SIZE / 2 metres). Such steps would place the true next location outside the predicted probability map, causing an index error in the loss function.

In [ ]:
step_df = filter_steps_by_window(
    step_df, window_size=WINDOW_SIZE, pixel_size=PIXEL_SIZE
)
print(f"Steps after window filter: {len(step_df):,}")

## `make_dataloaders()`
Splits step_df into train / val / test subsets (here 80 / 10 / 10 %) and returns a PyTorch DataLoader for each.

**The split is a `torch.utils.data.random_split`, not a chronological one** — each subset is a random sample of steps drawn from across the whole dataset and all 28 individuals. Consecutive steps are highly autocorrelated, so a random split leaks information between train and test and will make the validation numbers below look better than genuine out-of-sample performance. If you want an honest held-out estimate, split `step_df` yourself (by individual, or by time) and build the DataLoaders from the resulting subsets.

- Each DataLoader yields tuples of:
    * `x1`
        — spatial window tensor  (batch × n_spatial_channels × H × W)
        - One channel per raster *band* in LAYER_PATHS, cropped around each step's start pixel.
    * `x2`
        — scalar covariate tensor (batch × len(SCALAR_COLS))
        - Raw scalar values; broadcast into spatial maps *inside* the model.
    * `x3`
        — previous bearing tensor (batch × 1)
        - Conditions the movement kernel on the direction of the prior step.
    * `(px2, py2)`
        — pixel indices of the *true* next location within the window
        - Used by the loss function to look up the predicted probability at the observed destination.

The layer_paths argument causes the function to re-read the rasters internally, so the DataLoader is independent of the env_layers dict loaded above.

In [ ]:
dl_train, dl_val, dl_test = make_dataloaders(
    layer_paths=LAYER_PATHS,
    window_size=WINDOW_SIZE,
    batch_size=BATCH_SIZE,
    train_split=0.8,
    val_split=0.1,
    scalar_cols=SCALAR_COLS,
    df=step_df,
)
print(f"Train batches : {len(dl_train)}")
print(f"Val   batches : {len(dl_val)}")
print(f"Test  batches : {len(dl_test)}")

### Inspect one batch to verify tensor shapes match ModelParams expectations
`x1` has one channel per spatial *band* (NDVI + NDWI + NDMI = 3 with the index raster).  The scalar covariates in `x2` are broadcast to full H × W maps *inside* ConvJointModel and concatenated with `x1` there, giving the CNN N_SPATIAL_CHANNELS + len(SCALAR_COLS) = 3 + 5 = 8 input channels total.

The `px2` and `py2` are the target that the model is trying to predict, and are the pixel values of the next step (with the centre cell being (50, 50))

*`next(iter(dl_train))` will call the next batch, which are randomly shuffled, so calling the function again will provide different training samples.*

In [ ]:
x1, x2, x3, (px2, py2), _ = next(iter(dl_train))
print(f"Spatial   x1 : {x1.shape}   (batch × channels × H × W)")
print(f"Scalars   x2 : {x2.shape}   (batch × len(SCALAR_COLS))")
print(f"Bearing   x3 : {x3.shape}   (batch × 1)")
print(f"Next-step px2: {px2[:4].tolist()}  py2: {py2[:4].tolist()}")

### Plot the first set of tiles in the batch

In [ ]:
# Plot the spatial window "tiles" from the first batch: one row per sample, one
# column per raster band.  The centre pixel is the step's start location and the
# cross marks the true next location that the loss function will look up, so this
# is the quickest way to confirm the crops and targets line up with the rasters.
N_SHOW = 4  # how many samples from the batch to display

# Band labels straight from the GeoTIFF metadata, in the channel order the model sees
channel_labels = []
for name, arr in env_layers.items():
    if arr.ndim == 2:
        channel_labels.append(name)
        continue
    with rasterio.open(LAYER_PATHS[name]) as src:
        descs = src.descriptions
    for band in range(arr.shape[0]):
        channel_labels.append(descs[band] if descs and descs[band] else f"{name} b{band + 1}")

n_show = min(N_SHOW, x1.shape[0])
n_chan = x1.shape[1]
centre = WINDOW_SIZE // 2

fig, axes = plt.subplots(
    n_show, n_chan, figsize=(3.0 * n_chan, 3.0 * n_show), squeeze=False
)

for row in range(n_show):
    for col in range(n_chan):
        ax = axes[row, col]
        tile = x1[row, col].numpy()
        ax.imshow(tile, cmap="viridis", origin="upper")
        # Start location: always the centre cell of the window by construction
        ax.plot(centre, centre, "w+", markersize=9, markeredgewidth=1.5)
        # True next step — px2 is the column index, py2 the row index
        ax.plot(px2[row].item(), py2[row].item(), "rx", markersize=9, markeredgewidth=1.5)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(channel_labels[col], fontsize=10)
        if col == 0:
            ax.set_ylabel(
                f"sample {row}\n→ ({px2[row].item()}, {py2[row].item()})", fontsize=8
            )

fig.suptitle(
    f"First batch: {WINDOW_SIZE}×{WINDOW_SIZE} tiles  "
    "(white + = step start, red × = observed next step)",
    y=1.0,
)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "batch_tiles.png"), dpi=100, bbox_inches="tight")
plt.show()


## `ModelParams` 
A thin dict-like container (it is a wrapper around something else rather than storing data) that groups all architecture hyper-parameters so they can be passed to ConvJointModel and logged together.

This is where the parameters that are passed to the model are set.


In [ ]:
params = ModelParams({
    "batch_size":                BATCH_SIZE,
    "image_dim":                 WINDOW_SIZE,           # spatial window size (pixels)
    "pixel_size":                PIXEL_SIZE,            # metres per pixel
    "dim_in_nonspatial_to_grid": len(SCALAR_COLS),      # scalar cols tiled into window
    "dense_dim_in_nonspatial":   len(SCALAR_COLS),      # scalar inputs to the dense MLP
    "dense_dim_hidden":          64,                    # hidden units in the scalar MLP
    "dense_dim_in_all":          DENSE_DIM,             # flattened CNN output size
    # One channel per raster band (not per layer) + the tiled scalar covariates
    "input_channels":            N_SPATIAL_CHANNELS + len(SCALAR_COLS),
    "output_channels":           OUTPUT_CHANNELS,       # feature maps per conv layer
    "kernel_size":               3,                     # conv kernel spatial size
    "stride":                    1,
    "kernel_size_mp":            2,                     # max-pool kernel size
    "stride_mp":                 2,
    "padding":                   1,                     # same-padding preserves spatial
    "num_movement_params":       12,                    # movement kernel parameters
    "dropout":                   0.0,
    "device":                    DEVICE,
})
print(f"input_channels: {params.input_channels}  (must match x1 channels + len(SCALAR_COLS))")

## `ConvJointModel` 
The deepSSF model! A joint habitat–movement model with two heads:
1. Habitat head 
    — a small CNN applied to the spatial window (env layers + tiled scalars) that outputs a per-pixel log-suitability map.
2. Movement head 
    — a dense MLP (fully-connected layer) that takes the scalar covariates and the previous bearing, and produces parameters for a parametric movement kernel (a mixture of wrapped distributions centred on the current location).

The two log-probability surfaces are summed pixel-wise to give the joint predicted probability map for the animal's next step.

In [ ]:
model = ConvJointModel(params).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters : {n_params:,}")

## `negativeLogLikeLoss` 
Scores each predicted probability map against the true next-step pixel index; minimising it pushes probability mass toward observed locations.

In [ ]:
loss_fn = negativeLogLikeLoss(reduction="mean")

## `make_optimisers()`
Creates two Adam optimisers with separate learning rates:

- `lr_habitat`
    * controls the CNN (habitat) head
- `lr_movement`
    * controls the movement branch (`conv_movement` + `fcn_movement_all`), kept lower for stability

It also returns a `ReduceLROnPlateau` scheduler for each optimiser that reduces the learning rate by a factor of `scheduler_factor` (default = 0.1) if that head's validation loss does not improve for `scheduler_patience` epochs (default = 5).

Note that each scheduler now watches **its own** component of the validation loss. Previously both watched the combined loss, which is dominated by the movement component — so the movement head's plateau was what cut the habitat learning rate. With that fixed, the two learning rates no longer have to be separated by two orders of magnitude to stop movement from running away with training; start closer together and tune from there.


In [ ]:
optimisers, schedulers = make_optimisers(
    model, lr_habitat=1e-4, lr_movement=1e-5, scheduler_patience=3, scheduler_factor=0.5
)


## `EarlyStopping`
Monitors validation loss and saves the best model weights to disk whenever a new
minimum is reached.  Training stops automatically if the loss does not improve
for `patience` consecutive epochs.

### `monitor="both"` — when to *stop*
The combined loss is `habitat + movement + logZ`, and because the movement
surface is sharply peaked on the central pixels (animals mostly stay put between
steps) it improves by far more over a run than the diffuse habitat surface does —
roughly 1.2 nats against 0.05.  Habitat's improvement therefore sits *inside*
movement's epoch-to-epoch noise, and monitoring the combined loss alone ends the
run on movement's schedule while habitat is still learning.  With
`monitor="both"` each component keeps its own patience budget and training stops
only once both have plateaued.

### `checkpoint_on` — when to *save*
`logZ = logsumexp(habitat + movement)` is the normaliser of the joint surface.
Both surfaces arrive already normalised over the window, so it is the log of

```
Z = sum over cells of  p_habitat(cell) * p_movement(cell)
```

— the **overlap** between the two, i.e. the average habitat probability weighted
by where the movement kernel says the animal could actually go.  A flat habitat
surface gives `Z = 1/(H*W)`, hence `logZ = -log(H*W) = -8.64` for a 75 px window —
which is why an untrained model reports `total ≈ movement` almost exactly: the
uniform habitat term and the normaliser cancel.

`logZ` is what makes this a *step selection* likelihood rather than a plain
habitat model. But it also means **the combined loss is not monotone in the
habitat loss**. In the habitat-only stage of the run below, movement is pinned
and yet:

| epoch | combined | habitat | movement |
| --- | --- | --- | --- |
| 11 | 4.1155 | 8.5162 | 4.1268 |
| 12 | 4.1156 | **8.4817** | 4.1268 |
| … | | | |
| 20 | **4.1050** | 8.5781 | 4.1268 |

The combined loss falls every epoch, so a checkpoint is written every epoch —
but habitat was at its best back at epoch 12 and is *getting worse*.

#### Why, given the filters apply the same operation everywhere?
They do — `conv_habitat` is four `Conv2d` layers with no positional input (the
scalar covariates are broadcast to *constant* layers), so it is
translation-equivariant: identical covariates get an identical value wherever
they sit in the window.  It cannot single out "near the pig".

What differs is the **availability set** each criterion contrasts the used cell
against.  Writing `p_habitat = softmax(f)`, the habitat gradients are

```
combined :  E_q[df/dθ]        - df(y)/dθ      q = p_habitat * p_movement / Z
habitat  :  E_p_habitat[df/dθ] - df(y)/dθ
```

Both raise `f` at the observed cell and lower it at cells drawn from an
availability distribution — pure operations in *covariate* space. Only the
distribution differs:

| available ~ | effective cells (of 5625) | mean covariate distance from used cell |
| --- | --- | --- |
| `q` — the reachable set | 109 | 0.026 |
| `p_habitat` — the whole window | 5460 | 0.059 |

So the combined loss asks the filters to separate the chosen cell from
near-identical neighbours; the habitat loss asks them to separate it from the
whole 1500 m window.  Filters tuned for one are not the filters that win the
other.  `logZ` falling is a *symptom* of sharpened local discrimination — one
chosen cell up, its ~108 unchosen neighbours down — not a loophole.

Which criterion you want is a genuine choice:

- `checkpoint_on="total"` (default) saves the best **joint step-selection
  model**, with habitat fitted against availability. This is the classical iSSF
  estimand, and the right one if the habitat surface will be mapped as a
  *selection* surface with `predict_habitat_landscape()`.
- `checkpoint_on="active"` keys the checkpoint on whichever head is training
  during a single-head stage. It answers a different, RSF-flavoured question —
  which surface best predicts used locations against uniform availability over
  the window — which makes it most useful as a **diagnostic** of whether the
  habitat head is learning at all.

Keep the magnitudes in view: habitat here sits only ~0.04–0.15 nats below a
uniform surface while movement is ~4.4 nats below it, so this choice moves a
component worth a few percent of the model's predictive power.

### `head_paths` — keeping both
Checkpoints always hold the **whole** model — the heads are not stored
separately.  `head_paths` writes an *extra* file per head, each a complete
model, captured at the epoch where that head's own loss was lowest, tracked
across the entire run.  `load_head_weights()` then copies just one head's
weights out of such a file.

In [ ]:
early_stop = EarlyStopping(
    patience=10,
    verbose=True,
    monitor="both",                 # stop only when BOTH heads have plateaued
    path=str(OUTPUT_DIR / "best_model.pt"),
    checkpoint_on="active",         # a habitat-only stage saves on habitat loss
    # Also keep each head's own best epoch, across the whole run
    head_paths={
        "habitat":  str(OUTPUT_DIR / "best_model_habitat.pt"),
        "movement": str(OUTPUT_DIR / "best_model_movement.pt"),
    },
)

## `fit()`
Runs the full training loop.

- Inputs:
    * `model`
        - the name of the deepSSF model specified using `ConvJointModel` above
    * `image_trim_pixels`
        - how many outer pixel rings to trim from the output images before saving to prevent artifacts with the colour scaling due to the image padding being pulled into the image during the convolution layers.
    * `dl_train`
        - train dataloader (contains the training data)
    * `dl_val`
        - validation dataloader (contains the validation data used to assess the learning rate and early stopping)
    * `loss_fn`
        - loss function
    * `optimisers`
        - optimisers used to update the model weights
    * `schedulers`
        - learning-rate schedulers (used to update the learning rates)
    * `stages`
        - optional coordinate-ascent schedule; each stage names which sub-networks receive updates (see below)
    * `early_stopping`
        - stores the validation loss and stops the model training if it doesn't improve for a given number of epochs (set using `patience` in the `EarlyStopping` definition)
    * `snapshot_dir`
        - where to save images of the training progress
    * `snapshot_item`
        - which sample (movement step) from the validation dataset to run the model on to generate the movement, habitat selection and next-step probability surfaces

- Outputs:
    * history dict with keys `train_losses`, `val_losses`, `val_habitat_losses`, `val_movement_losses` and `stage`

**Index 0 of every history list is a pre-training baseline** — a validation pass run before the first epoch, with `train_losses[0] = nan` and `stage[0] = -1`. An untrained habitat surface is uniform over the window, so its baseline should land near `log(75 * 75) = 8.635`; the gap between that and the final `val_habitat_losses` is a direct read on how much the habitat head actually learned.

### Staged training

`stages` runs the two sub-networks as a coordinate ascent on the same joint likelihood, rather than reweighting it:

1. **movement only** — get a sensible availability kernel in place first, since habitat is fitted *conditional* on it
2. **habitat only** — habitat now runs to its own convergence, on its own early-stopping clock, with the movement surface held fixed in the likelihood as an offset
3. **both** — a joint pass to settle

A frozen sub-network keeps contributing its surface to the loss; only its weights stop moving. This is why staging is used instead of `negativeLogLikeLoss(freeze_movement=True)` — that flag *drops* the movement surface from the combined loss, so habitat would have to explain the movement structure too and would learn a distance-decay bump centred on the previous location, which is the opposite of a clean resource-selection surface.

Early stopping ends the current *stage* and advances to the next, so the final stage's early stop ends the run.

Every epoch, a plot of the training and validation loss, and the predicted probability maps (movement, habitat selection and next-step) are written to `SNAPSHOT_DIR`, which will be turned into a GIF in the next cell.


In [ ]:
history = fit(
    model=model,
    image_trim_pixels=4,
    window_size=WINDOW_SIZE,
    dl_train=dl_train,
    dl_val=dl_val,
    loss_fn=loss_fn,
    optimisers=optimisers,
    schedulers=schedulers,
    stages=[
        {"epochs": 5, "train": ("movement",)},   # availability kernel first
        {"epochs": 30, "train": ("habitat",)},    # habitat to its own convergence
        {"epochs": 20, "train": ("habitat", "movement")},  # joint settle
    ],
    early_stopping=early_stop,
    snapshot_dir=str(SNAPSHOT_DIR),
    snapshot_item=250,
    batch_size=BATCH_SIZE,
)


## `create_gif()`
Assembles the snapshots in `snapshot_dir` into an animation so you can see how
the predictions evolve as training progresses.

Display it with `IPython.display.Image`, which reads the file and embeds the
bytes in the notebook's output.  A hand-written `<img src="outputs/....gif">`
tag does *not* work here: the notebook's HTML output is rendered in a sandboxed
webview whose base URL is not the notebook's directory, so a relative path
resolves to nothing and you get an empty box.  Nothing is generating a still
frame — there is simply no image being loaded at all.  Embedding also means the
animation survives being exported to HTML or shared as a standalone `.ipynb`.

In [ ]:
gif_path = str(OUTPUT_DIR / "training_progress.gif")
create_gif(str(SNAPSHOT_DIR), gif_path, fps=5)
print(f"Training GIF → {gif_path}")

# Image() embeds the file's bytes in the output, so it renders wherever the
# notebook is opened.  Leave it as the last expression to display it.
Image(filename=gif_path)

In [ ]:
# Three panels rather than one: the habitat and movement components move on very
# different scales, so sharing an axis would flatten the habitat curve into a
# horizontal line.  x = 0 is the pre-training baseline recorded by fit().
uniform_nll = np.log(WINDOW_SIZE ** 2)
epochs = np.arange(len(history["val_losses"]))
stages = np.asarray(history["stage"])

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), constrained_layout=True)

panels = [
    ("Combined", history["val_losses"], history["train_losses"]),
    ("Habitat", history["val_habitat_losses"], None),
    ("Movement", history["val_movement_losses"], None),
]

for ax, (title, val, train) in zip(axes, panels):
    if train is not None:
        ax.plot(epochs, train, color="0.6", lw=1.5, label="train")
    ax.plot(epochs, val, color="C0", lw=1.5, label="val")
    if title == "Habitat":
        # A uniform surface over the window scores log(H*W), so this line is the
        # "learned nothing" mark — the whole question for the habitat head is how
        # far below it the curve gets.  Movement sits far below it from the first
        # epoch, and drawing it there would only compress that panel's axis.
        ax.axhline(uniform_nll, color="0.4", ls="--", lw=1,
                   label=f"uniform ({uniform_nll:.2f})")
    # Shade the staged-training blocks so the schedule is readable off the curve.
    for s in np.unique(stages[stages >= 0])[1::2]:
        within = epochs[stages == s]
        ax.axvspan(within.min() - 0.5, within.max() + 0.5, color="0.92", zorder=0)
    ax.set_xlabel("Epoch (0 = untrained baseline)")
    ax.set_ylabel("NLL (nats)")
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Training history — combined loss and its two components")
plt.savefig(str(OUTPUT_DIR / "loss_history.png"), dpi=100)
plt.show()


## Choosing which checkpoint to use
`fit()` leaves the model holding its **final-epoch** weights, not the best ones,
so load a checkpoint before testing or simulating.

- `best_model.pt` — the model to use for **simulation**.  Both heads from one
  epoch, so the joint likelihood is the one that was actually measured.
- `best_model_habitat.pt` / `best_model_movement.pt` — each head at its own best
  epoch, tracked across the whole run.

Mixing them is a deliberate trade: taking the habitat head from a different
epoch gives the sharpest habitat surface to *map*, but the two heads then come
from different epochs and the joint model's likelihood is no longer what either
checkpoint recorded.  Use it for the landscape predictions below, not for
simulating trajectories.

In [ ]:
# The joint model — what the simulations further down should use
ckpt = load_checkpoint(str(OUTPUT_DIR / "best_model.pt"), model)
print(f"best_model.pt         saved on '{ckpt['checkpoint_criterion']}' "
      f"= {ckpt['checkpoint_score']:.6f}")

hab_ckpt = load_checkpoint(str(OUTPUT_DIR / "best_model_habitat.pt"))
print(f"best_model_habitat.pt  saved on '{hab_ckpt['checkpoint_criterion']}' "
      f"= {hab_ckpt['checkpoint_score']:.6f}")

# How much habitat was left on the table by the last epoch.  The two numbers
# above are different quantities (a combined loss and a habitat loss) and are
# not comparable directly, so read the gap off the history instead.
hab_hist = np.asarray(history["val_habitat_losses"])
best_epoch = int(np.nanargmin(hab_hist))
print(f"\nhabitat NLL: best {hab_hist[best_epoch]:.4f} at epoch {best_epoch}, "
      f"final {hab_hist[-1]:.4f} — a gap of {hab_hist[-1] - hab_hist[best_epoch]:+.4f} nats")
print(f"(a uniform surface over the window scores {np.log(WINDOW_SIZE ** 2):.4f} - "
      f"a gap of {hab_hist[-1] - np.log(WINDOW_SIZE ** 2):+.4f}.")

# To map the sharpest habitat surface instead, overwrite just that head:
# load_head_weights(str(OUTPUT_DIR / "best_model_habitat.pt"), model, "habitat")

# Test the model against withheld data

## Convert the NumPy rasters loaded earlier into PyTorch tensors.
Both validate_next_step_probs() and simulate_trajectory() call `get_landscape(month_index)` at each step to fetch the current environmental layers as a list of 2-D tensors.

With static (non-seasonal) rasters the month_index argument is unused. For seasonal data (e.g. monthly NDVI composites) you would index into a stack of layers here based on the current month.

In [ ]:
# validate_next_step_probs() and simulate_trajectory() both stack the list returned
# by get_landscape() along a new channel dimension, so every element must be a *2-D*
# tensor.  Unbind the band dimension of any multi-band layer so the channel order
# matches what MovementDataset fed the model during training.
landscape_tensors = []
for arr in env_layers.values():
    t = torch.from_numpy(np.asarray(arr).astype(np.float32))
    landscape_tensors.extend([t] if t.ndim == 2 else list(t))

print(f"Landscape channels: {len(landscape_tensors)}  (each {tuple(landscape_tensors[0].shape)})")

def get_landscape(_month_index):
    # Returns one 2-D tensor per spatial channel, in the same order as LAYER_PATHS.
    return landscape_tensors

# Take the last 10 % of steps as a test sample.  CAVEAT: make_dataloaders() uses a
# random_split, so these steps were mostly *seen during training* — the numbers below
# are not a clean out-of-sample estimate.  Because step_df is ordered by individual
# name, this slice is also just the last few pigs alphabetically rather than the
# latest steps in time.  Hold out individuals or a date range explicitly for a real test.
n_test      = int(len(step_df) * 0.1)
test_sample = step_df.iloc[-n_test:].reset_index(drop=True)
print(f"Testing on {len(test_sample):,} steps")

## `validate_next_step_probs()`
Runs the model forward on each test step and records the probability mass assigned to the *true* next location.

- **Inputs**: 
    * `model`
    * `test_sample`
        - test dataframe that contains movement steps (from the data loaded above)
     * `get_landscape`
        - retrieves the landscape layers (spatial covariates) that were used to train the model
    * `transform`
        - raster Affine transform
    * `window_size`
        - window size of the local availability window (should be the same as for training)
    * `scalar_cols`
        - scalar column names (should be set from above)
    * `yday_col`
        - which column has the yday column to ensure the correct spatial layers (if dynamic through time) are indexed
    * `month_index_fn`
        - a function mapping day-of-year → month index (used by the `get_landscape` function). Here we are using static spatial layers so we can set it to 0.

- **Outputs**: 
    * `test_sample` dataframe with three new columns appended:
        - habitat_prob   — probability at the true location from the habitat head alone
        - move_prob      — probability at the true location from the movement head alone
        - next_step_prob — joint (habitat × movement) probability at the true location

Higher values mean the model performed better (there was higher predicted probability at the observed next-step). 


In [ ]:
# Use the last 10 % of steps as the test set (matches the DataLoader split)
n_test      = int(len(step_df) * 0.1)
test_sample = step_df.iloc[-n_test:].reset_index(drop=True)
print(f"Validating on {len(test_sample):,} steps")

val_results = validate_next_step_probs(
    model,
    test_sample,
    get_landscape=get_landscape,
    transform=raster_transform,
    window_size=WINDOW_SIZE,
    scalar_cols=tuple(SCALAR_COLS),
    yday_col="yday_t1",
    bearing_col="bearing_tm1",
    month_index_fn=lambda _yday: 0,   # static layers — month doesn't matter
)
print(val_results[["habitat_prob", "move_prob", "next_step_prob"]].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col, title in zip(
    axes,
    ["habitat_prob", "move_prob", "next_step_prob"],
    ["Habitat", "Movement", "Next-step"],
    strict=False,
):
    data = val_results[col].dropna()
    ax.hist(data, bins=40, edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("Probability")
axes[0].set_ylabel("Count")
plt.suptitle("Next-step probability distributions (test set)", y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "validation_probs.png"), dpi=100)
plt.show()

# Habitat selection across the full landscape

## `predict_habitat_landscape()`
Everything so far has run the model on a `WINDOW_SIZE` crop centred on the
animal.  But the habitat sub-network is *fully convolutional* — four 3x3
convolutions with stride 1 and padding 1, no pooling and no dense layer — so the
same trained filters can be swept across the entire raster in a single pass.
Convolution is translation-equivariant, so the value at a pixel is exactly what
the model would have produced for a window centred on that pixel.

That gives a landscape-wide habitat selection surface.  Read it as a **relative
ranking of cells**: "if the animal were somewhere on this landscape, how
attractive is each cell". It deliberately says nothing about whether the animal
could *reach* a cell in one step — that constraint lives in the movement kernel,
which takes no part here.  The two are only combined over a local window, which
is what `simulate_next_step()` does.

- **Inputs**:
    * `model`
        - the trained model; only `model.conv_habitat` is used
    * `landscape_rasters`
        - the spatial covariates as a list of 2-D tensors, in the same order and
          on the same scaling as training — i.e. exactly what `get_landscape()`
          returns
    * `scalars`
        - the scalar covariates for the moment being predicted, in `SCALAR_COLS`
          order.  `make_simulation_inputs()` builds this row for us, so the
          encoding cannot drift from what the simulation uses.  Each value is
          broadcast to a constant layer covering the whole landscape (the same
          thing `Scalar_to_Grid_Block` does for a window — that block can't be
          reused here because its output size is fixed to `WINDOW_SIZE`)
    * `normalise`
        - subtract the log-sum-exp over the whole landscape, making the surface
          a log-probability that sums to 1 across all valid cells
    * `edge_buffer`
        - how many border pixels to mask as `NaN`.  Defaults to
          `habitat_edge_buffer(model)` — the exact width the zero-padding
          reaches, 1 pixel per conv layer, so 4 here.  Those pixels saw padding
          rather than landscape and are not comparable with the interior
    * `chunk_rows`
        - process this many raster rows at a time.  A full-landscape activation
          is several hundred MB, enough to exhaust a GPU; the chunks overlap by
          `edge_buffer` so the result is identical to a single pass

- **Outputs**: a `[height, width]` NumPy array of log-probabilities, `NaN` in the
  border.  Take `np.exp()` for probabilities.

In [ ]:
from deepssf import (
    habitat_edge_buffer,
    make_simulation_inputs,
    predict_habitat_landscape,
)

# Predict for one moment in time: 2 am on day 200.  make_simulation_inputs
# returns one row per step, so ask for a single step and take row 0 — that
# guarantees the sin/cos encoding matches what the simulation feeds the model.
PRED_YDAY, PRED_HOUR = 200, 2.0
scalars_pred = make_simulation_inputs(
    n_steps=1, starting_yday=PRED_YDAY, starting_hour=PRED_HOUR
)[0][0]
print(f"scalar covariates ({', '.join(SCALAR_COLS)}):\n  {scalars_pred}")

habitat_log = predict_habitat_landscape(
    model,
    landscape_rasters=get_landscape(0),   # static layers → month index 0
    scalars=scalars_pred,
)
print(f"\nSurface {habitat_log.shape}, masked border "
      f"{habitat_edge_buffer(model)} px")
print(f"log-probability range: {np.nanmin(habitat_log):.2f} to "
      f"{np.nanmax(habitat_log):.2f}")
print(f"probabilities sum to  : {np.nansum(np.exp(habitat_log)):.4f}")

### Plot the landscape-scale habitat selection surface
Plotted on the **log** scale.  Across 9.5 million cells the probabilities are
tiny (a uniform surface would give ~1e-7 per cell) and a handful of very
attractive cells dominate the linear range, so a linear colour scale washes the
whole map out.  The colour limits are clipped to percentiles for the same
reason.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Percentile limits: a few extreme cells otherwise dominate the colour range
vmin, vmax = np.nanpercentile(habitat_log, [1, 99.9])

rasterio.plot.show(
    habitat_log, transform=raster_transform, ax=axes[0],
    cmap="viridis", vmin=vmin, vmax=vmax,
)
axes[0].set_title(f"Habitat selection (log prob) — yday {PRED_YDAY}, "
                  f"{PRED_HOUR:.0f}:00")

rasterio.plot.show(
    habitat_log, transform=raster_transform, ax=axes[1],
    cmap="viridis", vmin=vmin, vmax=vmax,
)
axes[1].plot(
    step_df["x1_"], step_df["y1_"],
    "r.", alpha=0.25, markersize=0.6, label="observed GPS fixes",
)
axes[1].legend(loc="upper right", markerscale=10)
axes[1].set_title("with observed locations overlaid")

for ax in axes:
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")

# One colourbar for both panels
fig.colorbar(axes[0].images[0], ax=axes, shrink=0.8, label="log probability")
plt.savefig(str(OUTPUT_DIR / "habitat_landscape.png"), dpi=120, bbox_inches="tight")
plt.show()

### How does habitat selection change through the day?
The scalar covariates are inputs to the habitat CNN, so the *same* landscape
produces a different surface at different times of day. Re-running the prediction
at different times of day shows what the model has learned.

Each surface is normalised over the landscape independently, so the panels show
where selection is concentrated at that hour, not the overall level of activity.

These surfaces **only** show the habitat selection, and not where would expect to
find pigs, as these maps do not take movement into consideration. To generate 
distribution maps we need to simulate movement trajectories. 

In [ ]:
PRED_HOURS = [0, 6, 12, 18]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
surfaces = {}
for ax, hour in zip(axes.flat, PRED_HOURS, strict=True):
    scalars_h = make_simulation_inputs(
        n_steps=1, starting_yday=PRED_YDAY, starting_hour=float(hour)
    )[0][0]
    surf = predict_habitat_landscape(
        model, landscape_rasters=get_landscape(0), scalars=scalars_h
    )
    surfaces[hour] = surf

    lo, hi = np.nanpercentile(surf, [1, 99.9])
    rasterio.plot.show(
        surf, transform=raster_transform, ax=ax, cmap="viridis", vmin=lo, vmax=hi
    )
    ax.set_title(f"{hour:02d}:00")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")

plt.suptitle(f"Habitat selection through the day (yday {PRED_YDAY})")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "habitat_landscape_by_hour.png"), dpi=120)
plt.show()

# How much does the surface actually move between hours?  If these differences
# are ~0 the model has not learned any diel variation in habitat use.
ref = surfaces[PRED_HOURS[0]]
for hour in PRED_HOURS[1:]:
    diff = np.nanmax(np.abs(surfaces[hour] - ref))
    print(f"max |log-prob difference| between 00:00 and {hour:02d}:00: {diff:.3f}")

### Create a GIF of the hourly habitat selection surfaces

Save each hourly habitat selection map as a PNG (within a subdirectory called 'habitat_selection_maps'), and then combine them into a GIF. 

In [ ]:
HABITAT_MAP_DIR = OUTPUT_DIR / "habitat_selection_maps"
HABITAT_MAP_DIR.mkdir(exist_ok=True)

# Predict all 24 hours first, then draw them.  The two passes are what lets every
# frame share one colour scale: with per-frame percentile limits (as in the 2x2
# figure above) each surface is re-normalised to its own range, so the animation
# flickers without showing where selection has actually moved.  At this raster
# size a surface is ~9 MB, so a full day is ~215 MB in memory.
hourly_surfaces = {}
for hour in range(24):
    scalars_h = make_simulation_inputs(
        n_steps=1, starting_yday=PRED_YDAY, starting_hour=float(hour)
    )[0][0]
    hourly_surfaces[hour] = predict_habitat_landscape(
        model, landscape_rasters=get_landscape(0), scalars=scalars_h
    )
    print(f"  {hour:02d}:00 done", end="\r")

# Shared colour limits over the whole day, clipped to the same percentiles used
# for the single-hour map (a handful of extreme cells otherwise take the range).
VMIN = min(np.nanpercentile(s, 1) for s in hourly_surfaces.values())
VMAX = max(np.nanpercentile(s, 99.9) for s in hourly_surfaces.values())
print(f"shared colour limits: {VMIN:.2f} to {VMAX:.2f}")

for hour, surf in hourly_surfaces.items():
    fig, ax = plt.subplots(figsize=(15, 12))
    rasterio.plot.show(
        surf, transform=raster_transform, ax=ax,
        cmap="viridis", vmin=VMIN, vmax=VMAX,
    )
    ax.set_title(f"Habitat selection (log prob) — yday {PRED_YDAY}, {hour:02d}:00")
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")
    fig.colorbar(ax.images[0], ax=ax, shrink=0.8, label="log probability")

    # Zero-padded hour so the filenames sort into clock order.  Note the absence
    # of bbox_inches="tight": it trims each figure to its own content, and frames
    # of differing pixel size cannot be stacked into a GIF.
    fig.savefig(HABITAT_MAP_DIR / f"habitat_hour_{hour:02d}.png", dpi=110)
    plt.close(fig)   # close, or 24 figures are drawn into the notebook output

print(f"Wrote 24 frames to {HABITAT_MAP_DIR}")

In [ ]:
import imageio.v2 as imageio

habitat_gif = OUTPUT_DIR / "habitat_selection_hourly.gif"

# Read the frames back off disk, sorted by filename — the zero-padded hour in the
# name puts them in clock order.  create_gif() (used for the training snapshots
# above) is deliberately not used here: it orders frames by an `epoch_<n>` field
# in the filename and, with no such field to key on, would leave them in whatever
# order the filesystem happened to list them.
frame_paths = sorted(HABITAT_MAP_DIR.glob("habitat_hour_*.png"))
frames = [imageio.imread(p) for p in frame_paths]

# duration is milliseconds *per frame*: half a second an hour 
# (`fps=` is accepted but deprecated in imageio v2 and warns.)
# loop=0 means loop forever rather than play once.
imageio.mimsave(habitat_gif, frames, duration=200, loop=0)
print(f"{len(frames)} frames → {habitat_gif} "
      f"({habitat_gif.stat().st_size / 1e6:.1f} MB)")

# As with the training GIF, Image() embeds the bytes so it renders wherever the
# notebook is opened.
Image(filename=str(habitat_gif))

### Save an hourly habitat selection surface as a raster (TIF)
Writing it out with the raster's transform and CRS means it can be opened
directly in QGIS/ArcGIS or read back with `rasterio`, and overlaid on the other
layers in `mapping/`.

In [ ]:
# The CRS is not returned by load_environmental_layers(), so read it off the
# source raster the transform came from.
with rasterio.open(next(iter(LAYER_PATHS.values()))) as src:
    raster_crs = src.crs

habitat_tif = OUTPUT_DIR / f"habitat_landscape_yday{PRED_YDAY}_hour{PRED_HOUR:.0f}.tif"
with rasterio.open(
    habitat_tif, "w",
    driver="GTiff",
    height=habitat_log.shape[0],
    width=habitat_log.shape[1],
    count=1,
    dtype="float32",
    crs=raster_crs,
    transform=raster_transform,
    nodata=np.nan,          # the masked edge buffer
    compress="deflate",
) as dst:
    dst.write(habitat_log.astype("float32"), 1)
    dst.set_band_description(1, "habitat selection log probability")

print(f"Wrote {habitat_tif} ({habitat_tif.stat().st_size / 1e6:.1f} MB)")

# Simulate new data

Select starting parameters for the model.

We've set the starting location to be the first location of the observed data.

In [ ]:
start_x = step_df["x1_"].iloc[10000]
start_y = step_df["y1_"].iloc[10000]
print(f"Starting location: ({start_x:.0f}, {start_y:.0f})")

# Simulate a trajectory

## `simulate_trajectory()` 
Generates a sequence of steps by iteratively:
1. Extracting a spatial window centred on the current location.
2. Running the model to obtain a joint log-probability surface over that window.
3. Sampling the next pixel from that surface (softmax-normalised probabilities).
4. Converting the sampled pixel back to projected coordinates and advancing time.

- **Inputs**: 
    * `model`
        - the trained model
    * `get_landscape`
        - retrieves the landscape layers (spatial covariates) that were used to train the model
    * `transform`
        - raster Affine transform
    * `start_x`
        - x coordinate of starting location
    * `start_y`
        - y coordinate of starting location
    * `n_steps`
        - number of steps
    * `starting_yday`
        - starting day-of-year 
    * `starting_hour`
        - starting hour-of-day
    * `time_between_steps`
        - time between steps in hours
    * `window_size`
        - size in pixels of the local landscape
    * `month_index_fn`
        - a function mapping day-of-year → month index (used by the `get_landscape` function). Here we are using static spatial layers so we can set it to 0.

- **Outputs**: DataFrame with one row per simulated step containing:
    * `x`, `y`           
        — projected coordinates of the simulated location
    * `hour`, `yday`     
        — time variables updated at each step
    * `month_index`
        - the month index used to index the spatial variables (will be 0 if using static spatial covariates)

In [ ]:
sim_df = simulate_trajectory(
    model,
    get_landscape=get_landscape,
    transform=raster_transform,
    start_x=start_x,
    start_y=start_y,
    n_steps=1000,
    starting_yday=test_sample["yday_t1"].iloc[0],
    starting_hour=test_sample["hour_t1"].iloc[0],
    time_between_steps=1.0,
    window_size=WINDOW_SIZE,
    month_index_fn=lambda _yday: 0,   # static layers
)
print(f"Simulated {len(sim_df)} steps")
sim_df.head()

### Plot the simulated data

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
# First spatial channel (NDVI with the index raster) as the background
rasterio.plot.show(
    landscape_tensors[0].numpy(), transform=raster_transform, ax=ax, cmap="viridis"
)
ax.plot(
    step_df["x1_"], step_df["y1_"],
    "w.", alpha=0.3, markersize=0.7, label="observed",
)
ax.plot(
    sim_df["x"], sim_df["y"],
    "r-o", markersize=1, linewidth=0.8, label="simulated",
)
ax.set_xlabel("Easting (m, projected)")
ax.set_ylabel("Northing (m, projected)")
ax.legend(markerscale=6)
ax.set_title("Simulated vs observed pig locations")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "simulated_trajectory.png"), dpi=100)
plt.show()

# Plot with an interactive map

## `plot_trajectories_folium()`
A static figure is fine for checking the overall shape of a trajectory, but not for
asking *where* the animal actually went.  This draws the simulated (and optionally
observed) locations over a satellite basemap so you can zoom in on individual steps.

- **Inputs**:
    * `trajectories`
        - a simulated-locations DataFrame.  If it has a `trajectory_id` column
          (as `simulate_trajectories()` returns) each id is drawn as its own
          toggleable line; a plain `sim_df` is drawn as a single line.
    * `observed`
        - observed locations to overlay, e.g. `step_df`.  Drawn as points rather
          than a line by default, because `step_df` holds 28 individuals and a
          single line would connect them with jumps that never happened.
    * `src_crs`
        - CRS of the input coordinates.  Everything is reprojected to EPSG:4326
          (lat/lon) for the web map — our data is in EPSG:3112.
    * `basemap`
        - `"esri_imagery"` (default), `"esri_topo"`, `"openstreetmap"`, or any
          XYZ tile-URL template.  See the folium docs for more:
          <https://python-visualization.github.io/folium/latest/user_guide/raster_layers/tiles.html>
    * `max_observed_points`
        - observed locations are thinned to this many points.  folium renders
          every point as its own element in the page, so all 100k+ fixes would
          lock up the notebook.
    * `save_path`
        - optionally also write a standalone HTML file.

- **Outputs**: a `folium.Map` — leave it as the last line of a cell to display it.

`folium` is an optional dependency: `pip install folium` (or
`pip install "deepSSF[maps]"`).

In [ ]:
from deepssf import plot_trajectories_folium

fmap = plot_trajectories_folium(
    sim_df,
    observed=step_df,
    src_crs="EPSG:3112",          # projected CRS of the GPS data and rasters
    basemap="esri_imagery",
    zoom_start=12,
    max_observed_points = 100000,
    save_path=str(OUTPUT_DIR / "simulated_trajectory_map.html"),
)
fmap

# Save the trajectory to a CSV

## `save_trajectories()`
Writes the simulated locations to `outputs/deepSSF_trajectories/`.  The filename
records how many trajectories, how many steps and the date, and the function
**never overwrites an existing file** — re-running the cell appends a `_2`, `_3`,
... run counter instead, so repeated runs accumulate rather than clobber.

- **Inputs**:
    * `trajectories` — a DataFrame (or list of DataFrames)
    * `output_dir` — created if it doesn't exist
    * `prefix` — leading part of the filename
    * `split` — write one file per `trajectory_id` instead of one combined file
- **Outputs**: the list of paths written

In [ ]:
from deepssf import save_trajectories

TRAJ_DIR = OUTPUT_DIR / "deepSSF_trajectories"

save_trajectories(sim_df, TRAJ_DIR, prefix="deepSSF_pigs")

# Simulating many trajectories

## `simulate_trajectories()`
Runs `n_trajectories` simulations and returns them stacked in one long-format
DataFrame with `trajectory_id` and `step` columns.

Three `method`s are available:

| `method` | how it works | when to use it |
| --- | --- | --- |
| `"batched"` *(default)* | every trajectory takes its step together, so the model sees a batch of `n_trajectories` windows per forward pass instead of one | almost always — the forward pass is the whole cost of the simulation, and it is far better used at batch size 50 than at batch size 1 |
| `"sequential"` | a plain loop over `simulate_trajectory()` | reference behaviour; needed if trajectories start at *different times* |
| `"parallel"` | the same loop across a thread pool | when start times vary and you want some overlap; the speed-up is modest (PyTorch releases the GIL inside its ops, but torch is already using several threads per op) and results are not reproducible from a seed |

The one restriction on `"batched"` is that all trajectories share a clock, so
`starting_yday` / `starting_hour` must be scalars.  Start *locations* can still
differ per trajectory — pass a list to `start_x` / `start_y` to release simulated
animals from different points.

In [ ]:
from deepssf import simulate_trajectories

N_TRAJECTORIES = 10
N_SIM_STEPS = 1000

sim_many = simulate_trajectories(
    model,
    get_landscape=get_landscape,
    transform=raster_transform,
    start_x=start_x,
    start_y=start_y,
    n_steps=N_SIM_STEPS,
    n_trajectories=N_TRAJECTORIES,
    method="batched",
    starting_yday=test_sample["yday_t1"].iloc[0],
    starting_hour=test_sample["hour_t1"].iloc[0],
    time_between_steps=1.0,
    window_size=WINDOW_SIZE,
    month_index_fn=lambda _yday: 0,   # static layers
)
print(f"{sim_many['trajectory_id'].nunique()} trajectories x "
      f"{N_SIM_STEPS} steps = {len(sim_many):,} locations")
sim_many.head()

### Releasing simulated animals from different starting points
Passing a sequence to `start_x` / `start_y` gives each simulated animal its own
release site.  Here we draw 10 observed locations at random and start one
trajectory from each — the same batched forward pass, so it costs nothing extra.

In [ ]:
# Draw starting locations from the observed fixes
release = step_df.sample(N_TRAJECTORIES, random_state=42)

sim_scattered = simulate_trajectories(
    model,
    get_landscape=get_landscape,
    transform=raster_transform,
    start_x=release["x1_"].to_numpy(),
    start_y=release["y1_"].to_numpy(),
    n_steps=1000,
    n_trajectories=N_TRAJECTORIES,
    method="batched",
    starting_yday=test_sample["yday_t1"].iloc[0],
    starting_hour=test_sample["hour_t1"].iloc[0],
    window_size=WINDOW_SIZE,
    month_index_fn=lambda _yday: 0,
)
print(f"{len(sim_scattered):,} locations from {N_TRAJECTORIES} release sites")

### Plot all of the simulated trajectories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, df, title in zip(
    axes,
    [sim_many, sim_scattered],
    ["Same release site", "Random release sites"],
    strict=True,
):
    rasterio.plot.show(
        landscape_tensors[0].numpy(), transform=raster_transform, ax=ax, cmap="viridis"
    )
    ax.plot(
        step_df["x1_"], step_df["y1_"],
        "w.", alpha=0.2, markersize=0.5, label="observed",
    )
    # One colour per simulated individual
    for tid, sub in df.groupby("trajectory_id"):
        ax.plot(sub["x"], sub["y"], "-", linewidth=0.7, alpha=0.9, label=f"sim {tid}")
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")
    ax.set_title(title)

plt.suptitle(f"{N_TRAJECTORIES} simulated trajectories")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "simulated_trajectories.png"), dpi=100)
plt.show()

## Create a heatmap of the simulated trajectories

Count the number of simulated locations per cell/pixel, with an option to discard the first *n* steps to reduce bias towards the initial parameters. 

This should be visualised as a PNG, but also stored as a raster such that it can be toggled on or off with the interactive map of the trajectories and observed data, and can be exported as a TIF.

In [ ]:
# --- Settings ---------------------------------------------------------------
HEATMAP_DF   = sim_scattered  # trajectories to summarise (or sim_many)
BURN_IN      = 0            # drop the first n steps of every trajectory
HEATMAP_AGG  = 4              # heatmap cell = HEATMAP_AGG x HEATMAP_AGG raster pixels


def trajectory_heatmap(
    df, transform, shape, burn_in=0, agg=1, x_col="x", y_col="y", step_col="step"
):
    """Count simulated locations per cell on a coarsened copy of the raster grid.

    *burn_in* discards the first n steps of each trajectory, which otherwise pile
    up around the release site(s) and reflect the starting conditions more than
    the fitted model.  *agg* aggregates agg x agg raster pixels into one heatmap
    cell: at the raster's native 20 m the counts are usually spread so thinly
    that almost every occupied cell holds a single location.

    Returns ``(counts, heatmap_transform)`` — the transform shares the rasters'
    origin, so the heatmap lines up with the environmental layers in QGIS.
    """
    if burn_in:
        df = df[df[step_col] >= burn_in]
    height, width = math.ceil(shape[0] / agg), math.ceil(shape[1] / agg)
    hm_transform = transform * rasterio.Affine.scale(agg)

    rows, cols = rasterio.transform.rowcol(
        hm_transform, df[x_col].to_numpy(), df[y_col].to_numpy()
    )
    rows, cols = np.asarray(rows), np.asarray(cols)
    inside = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
    if not inside.all():
        print(f"{(~inside).sum():,} locations fell outside the raster extent (dropped)")

    counts = np.zeros((height, width), dtype=np.int32)
    # np.add.at accumulates repeated (row, col) pairs; counts[rows, cols] += 1
    # would only ever add 1 per cell, no matter how many locations land in it.
    np.add.at(counts, (rows[inside], cols[inside]), 1)
    return counts, hm_transform


heatmap, heatmap_transform = trajectory_heatmap(
    HEATMAP_DF,
    raster_transform,
    landscape_tensors[0].shape,
    burn_in=BURN_IN,
    agg=HEATMAP_AGG,
)

print(f"Heatmap grid: {heatmap.shape[0]} x {heatmap.shape[1]} cells of "
      f"{PIXEL_SIZE * HEATMAP_AGG:.0f} m")
print(f"{heatmap.sum():,} locations counted after dropping the first {BURN_IN} steps "
      f"of each trajectory")
print(f"{(heatmap > 0).sum():,} occupied cells, up to {heatmap.max():,} locations in one cell")

### Plot the heatmap
Counts are drawn on a **log** colour scale — a handful of cells near the release
sites usually hold an order of magnitude more locations than the rest, which
would flatten a linear scale to a single bright dot.  Empty cells are masked so
the landscape shows through.

In [ ]:
from matplotlib.colors import LogNorm

fig, ax = plt.subplots(figsize=(8.5, 7))
# Greyscale background so the (colourful) heatmap stays readable on top
rasterio.plot.show(
    landscape_tensors[0].numpy(), transform=raster_transform, ax=ax, cmap="Greys_r"
)
img = ax.imshow(
    np.ma.masked_equal(heatmap, 0),  # empty cells stay transparent
    extent=rasterio.plot.plotting_extent(heatmap, heatmap_transform),
    cmap="magma",
    norm=LogNorm(vmin=1, vmax=max(int(heatmap.max()), 2)),
    interpolation="nearest",
    alpha=0.85,
)
ax.plot(
    step_df["x1_"], step_df["y1_"],
    "c.", alpha=0.15, markersize=0.5, label="observed",
)
fig.colorbar(img, ax=ax, label=f"simulated locations per {PIXEL_SIZE * HEATMAP_AGG:.0f} m cell")
ax.set_xlabel("Easting (m, projected)")
ax.set_ylabel("Northing (m, projected)")
ax.set_title(
    f"Simulated space use — {HEATMAP_DF['trajectory_id'].nunique()} trajectories, "
    f"steps {BURN_IN}+"
)
ax.legend(markerscale=8, loc="upper right")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "simulated_heatmap.png"), dpi=150)
plt.show()

### Save the heatmap as a raster (TIF)
Same treatment as the habitat selection surface above: writing the counts out
with the heatmap's own transform and the rasters' CRS gives a GeoTIFF that opens
directly in QGIS/ArcGIS.  `nodata=0` means unvisited cells are transparent
rather than a solid zero-valued block.

In [ ]:
# raster_crs was read off the source raster earlier; re-read it here so this
# section also runs if the habitat-surface cells were skipped.
with rasterio.open(next(iter(LAYER_PATHS.values()))) as src:
    raster_crs = src.crs

heatmap_tif = (
    OUTPUT_DIR
    / f"simulated_heatmap_{HEATMAP_DF['trajectory_id'].nunique()}traj"
      f"_{PIXEL_SIZE * HEATMAP_AGG:.0f}m_burnin{BURN_IN}.tif"
)
with rasterio.open(
    heatmap_tif, "w",
    driver="GTiff",
    height=heatmap.shape[0],
    width=heatmap.shape[1],
    count=1,
    dtype="int32",
    crs=raster_crs,
    transform=heatmap_transform,
    nodata=0,               # cells never visited
    compress="deflate",
) as dst:
    dst.write(heatmap, 1)
    dst.set_band_description(1, "simulated locations per cell")

print(f"Wrote {heatmap_tif} ({heatmap_tif.stat().st_size / 1e6:.2f} MB)")

### Toggle the heatmap on the interactive map
`plot_trajectories_folium()` returns the `folium.Map`, so the heatmap can be
added afterwards as an `ImageOverlay` — folium builds the layer control when the
map renders, so it picks up its own checkbox alongside the observed locations
and each simulated trajectory.

Leaflet works in lat/lon, so the counts are warped from the projected CRS onto
an EPSG:4326 grid first, then coloured here (rather than by folium) so that
unvisited cells can be made fully transparent.  `mercator_project=True` lets
folium re-sample that lat/lon image into the Web Mercator the basemap uses,
which is what keeps the heatmap aligned with the trajectories underneath it.

In [ ]:
import folium
from rasterio.warp import Resampling, calculate_default_transform, reproject

# --- Warp the counts to lat/lon ---
src_bounds = rasterio.transform.array_bounds(*heatmap.shape, heatmap_transform)
wgs_transform, wgs_width, wgs_height = calculate_default_transform(
    raster_crs, "EPSG:4326", heatmap.shape[1], heatmap.shape[0], *src_bounds
)
heatmap_wgs = np.zeros((wgs_height, wgs_width), dtype="float32")
reproject(
    source=heatmap.astype("float32"),
    destination=heatmap_wgs,
    src_transform=heatmap_transform,
    src_crs=raster_crs,
    dst_transform=wgs_transform,
    dst_crs="EPSG:4326",
    resampling=Resampling.nearest,  # nearest keeps the counts as counts
)
west, south, east, north = rasterio.transform.array_bounds(
    wgs_height, wgs_width, wgs_transform
)

# --- Colour it ourselves: same log scale as the PNG, alpha 0 where unvisited ---
rgba = plt.get_cmap("magma")(
    LogNorm(vmin=1, vmax=max(int(heatmap.max()), 2))(np.ma.masked_less(heatmap_wgs, 1))
)
rgba[..., 3] = np.where(heatmap_wgs >= 1, 0.75, 0.0)

fmap_heat = plot_trajectories_folium(
    HEATMAP_DF, 
    observed=step_df,
    src_crs="EPSG:3112",
    basemap="esri_imagery",
    zoom_start=11,
    max_observed_points=100000,
)
folium.raster_layers.ImageOverlay(
    image=rgba,
    bounds=[[south, west], [north, east]],
    name=f"heatmap (max {heatmap.max():,} per cell)",
    opacity=1.0,            # transparency is already baked into the alpha channel
    mercator_project=True,  # array is on a lat/lon grid; Leaflet draws Web Mercator
    show=True,
).add_to(fmap_heat)

heatmap_html = OUTPUT_DIR / "simulated_heatmap_map.html"
fmap_heat.save(str(heatmap_html))
print(f"Map saved: {heatmap_html}")
fmap_heat

### The same trajectories on the interactive map
Each trajectory gets its own colour and its own entry in the layer control (top
right), so individual animals can be switched on and off.

In [ ]:
plot_trajectories_folium(
    sim_many,
    observed=step_df,
    src_crs="EPSG:3112",
    basemap="esri_imagery",
    zoom_start=11,
    max_observed_points = 100000,
    save_path=str(OUTPUT_DIR / "simulated_trajectories_map.html"),
)

### Save them all
`split=False` (the default) writes one combined CSV with the `trajectory_id`
column; `split=True` writes one file per simulated individual.

In [ ]:
save_trajectories(sim_many, TRAJ_DIR, prefix="deepSSF_pigs")

# ...or one file per simulated animal:
# save_trajectories(sim_many, TRAJ_DIR, prefix="deepSSF_pigs", split=True)

### How much faster is `"batched"`?
Batching wins by more as `n_trajectories` grows, because a forward pass at batch
size 1 leaves most of the GPU (or most of the CPU's vector units) idle — running
50 trajectories together costs barely more per step than running one.  On an
Apple M-series GPU (`mps`), 200 steps with `WINDOW_SIZE = 75`:

| `n_trajectories` | `"batched"` | `"sequential"` | `"parallel"` | speed-up |
| --- | --- | --- | --- | --- |
| 4  | 1.5 s | 3.3 s  | 3.0 s  | 2.2x |
| 16 | 2.0 s | 14.1 s | 12.1 s | 6.9x |
| 64 | 1.6 s | 54.4 s | 48.9 s | 34x |

Note that `"parallel"` is only ~10% quicker than `"sequential"` here: torch is
already spreading each convolution across cores, so extra Python threads have
little left to win.  Worth re-running on your own hardware.

In [ ]:
import time

BENCH_TRAJ, BENCH_STEPS = 100, 100

for method in ("batched", "sequential", "parallel"):
    t0 = time.perf_counter()
    _ = simulate_trajectories(
        model,
        get_landscape=get_landscape,
        transform=raster_transform,
        start_x=start_x,
        start_y=start_y,
        n_steps=BENCH_STEPS,
        n_trajectories=BENCH_TRAJ,
        method=method,
        starting_yday=test_sample["yday_t1"].iloc[0],
        starting_hour=test_sample["hour_t1"].iloc[0],
        window_size=WINDOW_SIZE,
        month_index_fn=lambda _yday: 0,
    )
    elapsed = time.perf_counter() - t0
    print(f"{method:11s} {elapsed:6.2f} s "
          f"({BENCH_TRAJ * BENCH_STEPS / elapsed:7.0f} steps/s)")